# Ireland's Energy Price Shock: data preparation
Downloads and cleans two CSO tables:
- **CPM23**: EU Harmonised Index of Consumer Prices, monthly, Nov 1996 to present (core CSO High Value Dataset)
- **MEG01**: Metered Electricity Generation by fuel, half-hourly, Jan 2020 to Dec 2023 (supporting data)

In [1]:
import os
import pandas as pd

BASE_URL = "https://ws.cso.ie/public/api.restful/PxStat.Data.Cube_API.ReadDataset/{}/CSV/1.0/en"
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

for table in ["CPM23", "MEG01"]:
    path = f"data/raw/{table}.csv"
    if not os.path.exists(path):
        pd.read_csv(BASE_URL.format(table)).to_csv(path, index=False)
        print(f"Downloaded {table}")
    else:
        print(f"Using existing {path}")

Downloaded CPM23
Downloaded MEG01


## Clean CPM23 (household prices)
- Keep 10 price series.
- Keep the index (CPM23C01, base 2025 = 100) and the CSO's official annual rate of change (CPM23C03).
- Convert month codes (e.g. 202209) to dates.

In [2]:
SERIES = {
    "CP00":    "All items",
    "CP01":    "Food",
    "SA01":    "Energy",
    "SA15":    "All items excl. energy",
    "CP0451":  "Electricity",
    "CP04521": "Natural gas",
    "CP0453":  "Home heating oil",
    "CP0454":  "Solid fuels",
    "CP07221": "Diesel",
    "CP07222": "Petrol",
}
STATS = {"CPM23C01": "index", "CPM23C03": "yoy_pct"}

cpm = pd.read_csv("data/raw/CPM23.csv")
cpm = cpm[cpm["C04624V05409"].isin(SERIES) & cpm["STATISTIC"].isin(STATS)]

prices = (
    cpm.pivot(index=["TLIST(M1)", "C04624V05409"], columns="STATISTIC", values="VALUE")
       .rename(columns=STATS)
       .reset_index()
)
prices["date"]   = pd.to_datetime(prices["TLIST(M1)"].astype(str), format="%Y%m")
prices["series"] = prices["C04624V05409"].map(SERIES)
prices = prices[["date", "series", "index", "yoy_pct"]].sort_values(["series", "date"]).reset_index(drop=True)

print(prices.shape, "|", prices["date"].min().date(), "to", prices["date"].max().date())
print(prices.groupby("series").size())

(3590, 4) | 1996-11-01 to 2026-09-01
series
All items                 359
All items excl. energy    359
Diesel                    359
Electricity               359
Energy                    359
Food                      359
Home heating oil          359
Natural gas               359
Petrol                    359
Solid fuels               359
dtype: int64


## Clean MEG01 (electricity generation)
- MEG01 has two pre-calculated totals: "All time periods" (daily total) and "Net Generation" (total across fuels). Summing everything double-counts, so only "All time periods" rows are kept and "Net Generation" is the denominator.
- Daily totals are summed to months.
- Renewables = Wind, Solar, Renewable Hydro, Other Renewable.
- Excluded: Waste (only partly renewable), Biomass/Peat (mixed renewable and fossil), Pumped Storage Hydro and Battery Storage (store rather than generate electricity).

In [3]:
RENEWABLES = ["Wind", "Solar", "Renewable Hydro", "Other Renewable"]

meg = pd.read_csv("data/raw/MEG01.csv")
meg = meg[meg["Time Bands"] == "All time periods"].copy()
meg["date"] = pd.to_datetime(meg["TLIST(D1)"].str[:7], format="%YM%m")   # "2020M01D01" -> 2020-01-01

monthly = meg.groupby(["date", "Primary Fuel Output"])["VALUE"].sum().unstack()

renew = pd.DataFrame({
    "wind_mwh":      monthly["Wind"],
    "renewable_mwh": monthly[RENEWABLES].sum(axis=1),
    "total_mwh":     monthly["Net Generation"],
})
renew["renewable_pct"] = renew["renewable_mwh"] / renew["total_mwh"] * 100
renew["wind_pct"]      = renew["wind_mwh"] / renew["total_mwh"] * 100
renew = renew.reset_index()

print(renew.shape, "|", renew["date"].min().date(), "to", renew["date"].max().date())

(48, 6) | 2020-01-01 to 2023-12-01


In [4]:
y23 = renew[renew["date"].dt.year == 2023]
total_2023 = y23["total_mwh"].sum()
wind_2023  = y23["wind_mwh"].sum() / total_2023 * 100
renew_2023 = y23["renewable_mwh"].sum() / total_2023 * 100
dec_2023   = y23.loc[y23["date"].dt.month == 12, "renewable_pct"].iloc[0]

print(f"2023 total generation: {total_2023/1e6:.2f} million MWh   (CSO: 30.19)")
print(f"2023 wind share:       {wind_2023:.1f}%               (CSO: 38.9%)")
print(f"2023 renewable share:  {renew_2023:.1f}%")
print(f"Dec 2023 renewables:   {dec_2023:.1f}%               (CSO: 60%)")

assert abs(total_2023/1e6 - 30.19) < 0.05, "Total generation does not match CSO"
assert abs(wind_2023 - 38.9) < 0.1, "Wind share does not match CSO"
print("\nValidation passed")

2023 total generation: 30.19 million MWh   (CSO: 30.19)
2023 wind share:       38.9%               (CSO: 38.9%)
2023 renewable share:  44.0%
Dec 2023 renewables:   59.9%               (CSO: 60%)

Validation passed


In [5]:
prices_out = prices.round({"index": 2, "yoy_pct": 1})
renew_out  = renew.round({"wind_mwh": 0, "renewable_mwh": 0, "total_mwh": 0,
                          "renewable_pct": 1, "wind_pct": 1})

# CSV for submission
prices_out.to_csv("data/processed/energy_prices.csv", index=False)
renew_out.to_csv("data/processed/renewable_share.csv", index=False)

# JSON for pasting into the Vega-Lite editor
for df, name in [(prices_out, "energy_prices"), (renew_out, "renewable_share")]:
    df.assign(date=df["date"].dt.strftime("%Y-%m-%d")).to_json(
        f"data/processed/{name}.json", orient="records")

print("Saved:", sorted(os.listdir("data/processed")))

Saved: ['energy_prices.csv', 'energy_prices.json', 'renewable_share.csv', 'renewable_share.json']
